# 02 · Developing agentic pipelines on Union

We build an agent up in five steps, each adding one Flyte 2.0 capability:

1. **A durable LLM step**: `@flyte.trace` makes one model call observable and crash-proof.
2. **The agent loop, by hand**: a tool-use `while` loop, so the harness isn't magic.
3. **The harness**: `flyte.ai.agents.Agent` writes that loop for you.
4. **A parallel pipeline**: plan → fan out one worker per sub-topic → synthesize.
5. **A hosted chat UI**: serve the agent behind a web chat with one Flyte object.

**Why run agents on Flyte at all?** Research on multi-agent failures finds most
breakage is *orchestration*, not model IQ, nobody checkpoints state, retries
transient errors, or audits hand-offs. Flyte makes those four things infrastructure:

- **Durability**: typed, persisted artifacts replace in-process scratchpads.
- **Observable isolation**: each step is a traced span; tools can be isolated tasks.
- **Recovery**: retries / timeouts handle rate limits and network blips, not the LLM.
- **Fan-out**: `asyncio.gather` runs sub-agents in parallel, each its own action.

The full pipeline is [`../workflows/02_agentic_pipeline.py`](../workflows/02_agentic_pipeline.py).

Docs: [Build an agent](https://www.union.ai/docs/v2/flyte/user-guide/build-agent/) ·
[Pure Python agents](https://www.union.ai/docs/v2/flyte/user-guide/build-agent/building-agents/) ·
[traces](https://www.union.ai/docs/v2/flyte/user-guide/core-concepts/tracing/)

In [1]:
from pathlib import Path
import asyncio, json
import flyte, flyte.report

flyte.init_from_config(Path(".flyte") / "config.yaml")

MODEL = "claude-haiku-4-5"                 # litellm alias -> Anthropic (harness + traced calls)
MODEL_ANTHROPIC = "claude-haiku-4-5-20251001"  # exact id for the raw Anthropic SDK (step 2)

# One image for the notebook: litellm (harness + our traced LLM steps), anthropic
# (the hand-rolled step), and unionai-reuse (warm containers in step 4).
image = flyte.Image.from_debian_base().with_pip_packages("litellm", "anthropic", "unionai-reuse>=0.1.9")
anthropic_secret = flyte.Secret(key="ANTHROPIC_API_KEY", as_env_var="ANTHROPIC_API_KEY")

## Step 1 · A durable LLM step

Wrap a model call in a helper decorated with `@flyte.trace` (no parentheses). The
call still runs *inside* the task's container (no new pod) but its inputs and
outputs are **persisted and shown as a span** on the run page. If the task crashes
after this call, the traced result is not recomputed on replay.

In [2]:
from litellm import acompletion

step_env = flyte.TaskEnvironment(
    name="agent_step",
    image=image,
    resources=flyte.Resources(cpu=1, memory="1Gi"),
    secrets=[anthropic_secret],
)

@flyte.trace
async def llm(prompt: str) -> str:
    r = await acompletion(model=MODEL, messages=[{"role": "user", "content": prompt}])
    return r.choices[0].message.content

@step_env.task
async def summarize_url_idea(topic: str = "durable agents") -> str:
    return await llm(f"In two sentences, why do {topic} belong on a workflow engine?")

In [3]:
run = flyte.run(summarize_url_idea, topic="durable agents")
print("Run URL:", run.url)
run.wait(); run.outputs()

> Building 1 image...

> Building image flyte for environment agent_step

i Image flyte:de9e2b424cfd3dec60d70d94bc1b44aa was not found or has expired

> Image flyte:de9e2b424cfd3dec60d70d94bc1b44aa not found, building...

> Submitting a new build...

...

✓ Built image for environment agent_step: 356633062068.dkr.ecr.us-east-2.amazonaws.com/union/demo:flyte-de9e2b424cfd3dec60d70d94bc1b44aa

/Users/leonmenkreo/Documents/union/solutions-engineering/.venv/lib/python3.13/site-packages/rich/live.py:260: 
UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

Run URL: https://demo.hosted.unionai.cloud/v2/domain/development/project/leon-demo/runs/ucc95tvqz6t7r587v4f4


Run 'ucc95tvqz6t7r587v4f4' completed successfully.

ActionOutputs(o0="# Durable Agents on Workflow Engines

Durable agents maintain state and progress across interruptions, retries, and failures, making them ideal for workflow engines that must reliably orchestrate long-running, multi-step processes. By persisting their execution context, durable agents enable workflows to resume exactly where they left off after system crashes, timeouts, or deliberate pauses—eliminating the need to restart entire sequences from the beginning.")

## *Bonus* Step 2 · The agent loop, by hand

An agent is a loop: the model **reasons** about which tool to call, we **act** by
running it, and feed the **observation** back, until the model stops asking for
tools. Here it is explicitly, with the raw Anthropic SDK and native tool-use. The
`@env.task` is the durable unit; each **tool execution** is a `@flyte.trace` span.

In [ ]:
import anthropic

TOOLS = [
    {"name": "add", "description": "Add two numbers.",
     "input_schema": {"type": "object", "properties": {"a": {"type": "number"}, "b": {"type": "number"}}, "required": ["a", "b"]}},
    {"name": "multiply", "description": "Multiply two numbers.",
     "input_schema": {"type": "object", "properties": {"a": {"type": "number"}, "b": {"type": "number"}}, "required": ["a", "b"]}},
]

@flyte.trace                         # each tool call = a persisted span
async def run_tool(name: str, args: dict) -> str:
    result = {"add": lambda a, b: a + b, "multiply": lambda a, b: a * b}[name](**args)
    return str(result)

@step_env.task
async def calc_agent_manual(question: str = "What is (12 + 8) * 3?", max_steps: int = 5) -> str:
    client = anthropic.AsyncAnthropic()
    messages = [{"role": "user", "content": question}]
    for _ in range(max_steps):
        resp = await client.messages.create(
            model=MODEL_ANTHROPIC, max_tokens=1024, tools=TOOLS, messages=messages,
            system="You are a calculator. Use the tools; never do mental math.",
        )
        if resp.stop_reason != "tool_use":                       # model is done -> final answer
            return "".join(b.text for b in resp.content if b.type == "text")
        messages.append({"role": "assistant", "content": resp.content})
        results = []
        for b in resp.content:
            if b.type == "tool_use":
                out = await run_tool(b.name, b.input)             # act (traced)
                results.append({"type": "tool_result", "tool_use_id": b.id, "content": out})
        messages.append({"role": "user", "content": results})     # observation
    return "max steps reached"  # loop exhausted


In [ ]:
run = flyte.run(calc_agent_manual, question="What is (12 + 8) * 3?")
print("Run URL:", run.url)
run.wait(); run.outputs()

## Step 3 · The harness writes that loop for you

Everything in step 2 (the tool schema, the `while` loop, the tool-result plumbing)
is boilerplate. `flyte.ai.agents.Agent` does it. Tools are **plain functions**
(the harness reads the signature + docstring for the schema); you call
`await agent.run.aio(message)`.

| | Hand-rolled loop (step 2) | `flyte.ai.agents.Agent` (step 3) |
|---|---|---|
| **What you write** | a `while`/`for` loop in an `@env.task`, calling `@flyte.trace` `reason()`/`act()` | `Agent(model=..., tools=[fn, ...])` + `await agent.run.aio(msg)` |
| **You control** | exact prompts, tool schema, stop condition | instructions + tools; harness runs the loop |
| **Get for free** | full transparency (great for teaching / custom control) | tool-calling, memory, HITL, retries |
| **Durability** | `@flyte.trace` on each step | every step traced by the harness |

Both run on the same substrate: **durable checkpointing, sandboxed isolation,
parallel fan-out, and per-step observability**.

In [4]:
from flyte.ai.agents import Agent

async def add(a: float, b: float) -> float:
    "Add two numbers."
    return a + b
async def multiply(a: float, b: float) -> float:
    "Multiply two numbers."
    return a * b

calculator = Agent(
    name="calculator", model=MODEL,
    instructions="You are a calculator. Use the tools; never do mental math.",
    tools=[add, multiply], max_turns=6,
)

@step_env.task
async def calc_agent_harness(question: str = "What is (12 + 8) * 3?") -> str:
    result = await calculator.run.aio(question)
    return result.summary or result.error

In [5]:
run = flyte.run(calc_agent_harness, question="What is (12 + 8) * 3?")
print("Run URL:", run.url)
run.wait(); run.outputs()

> Building 1 image...

> Building image flyte for environment agent_step

✓ Built image for environment agent_step: 356633062068.dkr.ecr.us-east-2.amazonaws.com/union/demo:flyte-de9e2b424cfd3dec60d70d94bc1b44aa

Run URL: https://demo.hosted.unionai.cloud/v2/domain/development/project/leon-demo/runs/ursk6jvpk7c6jm7btwmm


Run 'ursk6jvpk7c6jm7btwmm' completed successfully.

ActionOutputs(o0="The answer is **60**.")

## Step 4 · A parallel, dynamic pipeline

The real shape: **plan** sub-topics at runtime, **fan out one worker per topic in
parallel**, then **synthesize**. Three Flyte 2.0 features carry it:

- **Traces**: `@flyte.trace` on `plan`, `synthesize`, and the shared `llm` helper
  makes every call a durable span on the run page, no extra pods.
- **Reusable containers**: `ReusePolicy` keeps the worker replicas warm, so the
  fan-out skips pod cold-starts (which dominate latency once every task is an LLM call).
- **Dynamic fan-out**: `plan` decides *how many* topics at runtime; we `gather` one
  worker task per topic inside a `flyte.group`.

A fuller, tool-using variant lives in [`../workflows/02_agentic_pipeline.py`](../workflows/02_agentic_pipeline.py).

In [6]:
from pydantic import BaseModel

# Reusable workers: warm replicas skip pod cold-starts on every fan-out call.
worker_env = flyte.TaskEnvironment(
    name="researcher", image=image, secrets=[anthropic_secret],
    resources=flyte.Resources(cpu=1, memory="1Gi"),
    reusable=flyte.ReusePolicy(replicas=(1, 2), idle_ttl=120, concurrency=4, scaledown_ttl=120),
)
# The driver fans out to workers, so it depends on them.
driver_env = flyte.TaskEnvironment(
    name="research_driver", image=image, secrets=[anthropic_secret],
    resources=flyte.Resources(cpu=1, memory="500Mi"), depends_on=[worker_env],
)

class ResearchResult(BaseModel):
    question: str
    topics: list[str]
    answer: str

# Two traced LLM steps + one warm worker. `llm` is the traced helper from step 1.
@flyte.trace
async def plan(question: str, n: int) -> list[str]:
    reply = await llm(f"List exactly {n} short research sub-topics for: {question}. "
                      "Reply as a JSON array of strings, nothing else.")
    return json.loads(reply[reply.index("["): reply.rindex("]") + 1])[:n]

@flyte.trace
async def synthesize(question: str, topics: list[str], findings: list[str]) -> str:
    notes = "\n".join(f"- {t}: {f}" for t, f in zip(topics, findings))
    return await llm(f"Question: {question}\nFindings:\n{notes}\n\n"
                     "Write a 4-sentence recommendation grounded in the findings.")

@worker_env.task
async def research_topic(topic: str) -> str:
    return await llm(f"In 3 sentences, give the key facts a decision-maker needs on: {topic}")

In [7]:
@driver_env.task(report=True)
async def research(question: str = "Should we expand into the EU market?", n: int = 3) -> ResearchResult:
    topics = await plan(question, n)                    # dynamic: the model picks the topics
    print("planned topics:", topics)

    with flyte.group("fan-out"):                        # one warm worker per topic, in parallel
        findings = await asyncio.gather(*[research_topic(t) for t in topics])

    answer = await synthesize(question, topics, findings)
    await flyte.report.replace.aio(
        f"<h2>{question}</h2>"
        + "".join(f"<h3>{t}</h3><p>{f}</p>" for t, f in zip(topics, findings))
        + f"<hr><h3>Recommendation</h3><p>{answer}</p>")
    await flyte.report.flush.aio()
    return ResearchResult(question=question, topics=topics, answer=answer)

In [8]:
run = flyte.run(research, question="Should we expand into the EU market?", n=3)
print("Run URL:", run.url)
run.wait()
run.outputs()

> Building 2 images...

> Building image flyte for environment research_driver

> Building image flyte for environment researcher

✓ Built image for environment research_driver: 356633062068.dkr.ecr.us-east-2.amazonaws.com/union/demo:flyte-de9e2b424cfd3dec60d70d94bc1b44aa

✓ Built image for environment researcher: 356633062068.dkr.ecr.us-east-2.amazonaws.com/union/demo:flyte-de9e2b424cfd3dec60d70d94bc1b44aa

Run URL: https://demo.hosted.unionai.cloud/v2/domain/development/project/leon-demo/runs/uktd72d4s8bgpr78szjc


Run 'uktd72d4s8bgpr78szjc' completed successfully.

ActionOutputs(o0=question='Should we expand into the EU market?' topics=['EU market size and growth potential compared to current markets', 'Regulatory compliance costs and barriers to entry in EU', 'Competitive landscape and market saturation in target EU regions'] answer='# Recommendation\n\n**Pursue selective EU expansion, but only in high-growth niches or underserved Central/Eastern European markets where regulatory compliance costs are justified by lower competition.** Western EU markets (Germany, France) are too saturated with entrenched competitors to justify the 3-5% revenue compliance burden for new entrants unless you have a differentiated offering in emerging sectors like sustainability or AI-driven services. **Prioritize markets with proportionally lower regulatory barriers and assess whether your company size and revenue base can absorb upfront legal and compliance infrastructure costs without compromising core operations.** If your competitive advantage is strong and your

Open the run page and read it top to bottom: the **`fan-out` group** with one
`research_topic` action per sub-topic (parallel, on warm workers), the `plan` and
`synthesize` calls as **traced spans**, and the **Report** tab with the
recommendation. A pipeline you can *inspect and replay*, not a black box.

## Step 5 · Ship it as a hosted chat UI

Union can host an agent behind a web chat with one object: `AgentChatAppEnvironment`
wraps a `flyte.ai.agents.Agent` and serves a chat at a URL. Each message runs through
a **durable task entrypoint**, so the agent's tool calls are the same traced,
replayable actions as the pipeline above; here the tool is a durable Flyte task, `research_topic`, defined right in this cell.

Docs: [Agent chat UI](https://www.union.ai/docs/v2/union/user-guide/agents/build-agent/agent-chat-ui/).

In [ ]:
# Self-contained: this cell defines its own env, tool, agent, and chat entrypoint,
# with no dependency on earlier cells. The agent's tool is a durable Flyte task on
# the SAME env the app depends on, so it is always attached when the chat serves.
from flyte.ai.agents import Agent
from flyte.ai.chat import AgentChatAppEnvironment, CustomTheme

MODEL = "claude-haiku-4-5"
chat_image = flyte.Image.from_debian_base().with_pip_packages("litellm", "fastapi", "uvicorn")
chat_secret = flyte.Secret(key="ANTHROPIC_API_KEY", as_env_var="ANTHROPIC_API_KEY")

chat_env = flyte.TaskEnvironment(
    name="research_chat",
    image=chat_image,
    resources=flyte.Resources(cpu=1, memory="1Gi"),
    secrets=[chat_secret],
)

# A tiny internal knowledge base + the durable tool the agent calls.
KB = {
    "market size": "The EU SaaS market is ~$95B, growing ~12% YoY. Germany, France, Nordics lead.",
    "revenue": "FY24 revenue was $12.4M, up 38% YoY. 22% of demo requests come from EU domains.",
    "competition": "Two incumbents hold ~40% EU share; both lack a self-serve tier, our wedge.",
    "compliance": "Selling into the EU needs GDPR data-residency; public-sector deals need EU-hosted infra.",
    "team": "45 employees, all US-based. No German/French sales; one engineer in Lisbon.",
    "pricing": "Pricing is USD-only; EU buyers expect EUR invoicing and net-30 terms.",
}

@chat_env.task
async def research_topic(topic: str) -> str:
    """Look up factual notes about a topic from the internal knowledge base."""
    tl = topic.lower()
    for key, note in KB.items():
        if key in tl or tl in key:
            return note
    return f"No internal notes on '{topic}'. Known topics: {', '.join(KB)}."

# The agent's one tool is that durable task (same env as the entrypoint below).
assistant = Agent(
    name="research-assistant", model=MODEL, max_turns=8,
    instructions="You are a market research assistant. Call research_topic to gather "
                 "facts before answering, then reply concisely.",
    tools=[research_topic],
)

# The task entrypoint owns the agent loop; the chat UI runs it once per message.
@chat_env.task(report=True)
async def chat_entrypoint(message: str, history: list[dict]) -> dict:
    result = await assistant.run.aio(message, memory=history)
    return {"summary": result.summary, "error": result.error,
            "attempts": result.attempts, "charts": [], "code": ""}

chat_app = AgentChatAppEnvironment(
    name="research-chat",
    agent=assistant,
    task_entrypoint=chat_entrypoint,
    title="Market research assistant",
    subtitle="A flyte.ai.agents.Agent backed by a durable Flyte task tool.",
    theme=CustomTheme(accent_color="#6F2AEF"),
    prompt_nudges=[
        {"label": "EU market", "prompt": "What should we know about the EU SaaS market?"},
        {"label": "Compliance", "prompt": "What compliance matters for selling into the EU?"},
    ],
    image=chat_image,
    resources=flyte.Resources(cpu=1, memory="1Gi"),
    secrets=[chat_secret],
    depends_on=[chat_env],   # the tool + entrypoint env, so the tool is attached at serve time
    requires_auth=True,      # set False for an anonymous public URL
    passthrough_auth=True,
)

# Deploy: Union hosts the chat at a URL.
handle = flyte.serve(chat_app)
print(handle.url)


## Recap

From one traced model call to a parallel, dynamic pipeline to a hosted chat agent,
every step stayed plain, inspectable Flyte:

- **`@flyte.trace`** turns any LLM or tool call into a durable span.
- **`ReusePolicy`** keeps fan-out workers warm.
- **`asyncio.gather` + `flyte.group`** fan out a runtime-decided number of workers.
- **`AgentChatAppEnvironment`** ships the agent behind a hosted chat UI.

**Next:** Part 3, [fix a broken workflow with your CLI agent](../workflows/03_fix_broken_workflow.py) ·
back to [01 · Best Practices](./01_best_practices.ipynb).